In [ ]:
import pandas as pd
import numpy as np

In [ ]:
'''
DataGuide에서 받은 전종목 일별 종가파일 읽어오기(공휴일포함/주말제외/csv UTF-8 형식으로)
헤더는 종목명으로 하고, 종목명 밑에 불필요한 항목들 제거, thousands 넣어서 쉼표 제거(thousand 안넣으면 처리하기 매우 복잡!!)
종목명이 아니라 종목번호로 헤더 쓴다면 header=8, skiprows = [9, 10, 11, 12, 13]
'''

df_daily = pd.read_csv('data/5y_closingprice.csv', header = 9, skiprows = [10,11,12,13], thousands = ',')

In [ ]:
# 인덱스를 종목일자로 변경
df_daily['Symbol Name'] = pd.to_datetime(df_daily['Symbol Name'])
df_daily = df_daily.set_index('Symbol Name')
df_daily.index.name = 'date'

In [ ]:
# 데이터프레임에 빈칸제거
df_daily.dropna(how='all', inplace = True)
df_daily.dropna(axis=1, how='all', inplace = True)

## 방법1 : 시계열데이터에서 일정간격 수익률 계산

* 방법 : 시계열 데이터에서 22일 간격으로 월간수익률 계산
* 주요 용도 : 역사적 수익률 및 베타 계산

In [ ]:
# 월간수익률 구하기(1개월은 workingday 22일이라 하고 계산, 22일 밀어내기 해서 원본에서 빼기)
month_len = 22
df_month_return = (df_daily - df_daily.shift(month_len))/df_daily

In [ ]:
# 밀어내기식으로 구하면서 발생하는 앞쪽의 빈칸 제거
df_month_return.dropna(how='all', inplace = True)
df_month_return

## 방법2 : Bootstrap 방식으로 일간수익률에서 월간수익률 계산

* 방법 : 일간수익률 풀에서 n개 뽑아 가상의 월간수익률 계산
* 주요 용도 : VaR 계산

In [ ]:
#일간수익률 계산
df_daily_return = (df_daily - df_daily.shift(1))/df_daily
df_daily_return.dropna(how='all', inplace = True)

In [ ]:
#일간수익률 샘플링해서 월간수익률 계산하는 함수 정의

def sampling_month_return(df, month_len):
    df1 = df.sample(n=month_len) # 종목별로 n개 일간수익률 샘플 추출
    temp = df1.sum() # 월간수익률을 일간수익률의 산술평균으로 계산(기하평균이 맞을지?)
    return pd.DataFrame(temp).transpose() # 데이터프레임 형태로 리턴

In [ ]:
#월간수익률 담을 df
df_month_return2 = pd.DataFrame([], columns = df_daily_return.columns)

# 월간수익률 10개 샘플링(한달을 워킹데이 22일로 생각)
for i in range(10):
    temp = sampling_month_return(df_daily_return, 22)
    df_month_return2 = df_month_return2.append(temp)
    
df_month_return2